In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import coremltools as ct
import tomllib


In [5]:
MODEL_NAME = "98.30%-5k-03-10-04:55"

In [ ]:
# 1. Rebuild the exact model architecture
class SitelenPonaCNN(nn.Module):
    def __init__(self, num_classes=137):
        super(SitelenPonaCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 256)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        # Note: We drop the dropout layer for the export, as it's only for training
        x = self.fc2(x)
        return x

# 2. Load the labels from TOML
TOML_PATH = "../tp-dataset/categories/typst.toml" # Adjust if necessary
with open(TOML_PATH, "rb") as f:
    data = tomllib.load(f)
GLYPHS = [filename.replace(".toml", "") for filename in data.get("wordlist", {}).get("list", [])]

# 3. Load the trained weights
model = SitelenPonaCNN(num_classes=len(GLYPHS))
model.load_state_dict(torch.load(f"../ocr-demo/models/{MODEL_NAME}.pth", map_location="cpu"))
model.eval() # Set to evaluation mode

# 4. Trace the model using a dummy 28x28 input
dummy_input = torch.rand(1, 1, 28, 28)
traced_model = torch.jit.trace(model, dummy_input)

# 5. Convert to CoreML
# We use ImageType. The scale=1/255.0 is CRITICAL because iOS feeds pixel values 0-255, 
# but our PyTorch model was trained on 0.0-1.0!
image_input = ct.ImageType(
    name="image", 
    shape=(1, 1, 28, 28), 
    color_layout=ct.colorlayout.GRAYSCALE, 
    scale=1/255.0
)

# Pass the labels so iOS outputs a nice dictionary instead of a raw array of floats
classifier_config = ct.ClassifierConfig(class_labels=GLYPHS)

mlmodel = ct.convert(
    traced_model,
    inputs=[image_input],
    classifier_config=classifier_config,
    minimum_deployment_target=ct.target.iOS16,
)

# 6. Save!
output_name = f"models/SitelenPonaModel.mlpackage"
mlmodel.save(output_name)
print(f"Successfully exported {output_name}!")

Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 3661.82 passes/s]


Successfully exported models/98.30%-5k-03-10-04:55.mlpackage!
